# Data Cleaning Task: Fitness Bookings

You are working with booking records exported from a fitness studio's
scheduling system. The studio offers classes such as Yoga, Pilates
HIIT and CrossFit across three locations (Downtown, Uptown, Westside).

The export was pulled directly from an old booking system and combines manual entries with online bookings, so it has **not** been cleaned
Your job is to explore the dataset, identify the data quality issues, and produce a clean,
analysis-ready version of it.


### Import necessary libraries

In [1]:
import pandas as pd

### Load data

In [2]:
df = pd.read_csv("uncleaned_fitness_bookings.csv")

### Understand the Dataset

- Check the shape of the dataset (how many rows and columns).
- Look at the first and last several rows to get a feel for the data.
- Use `.info()` to check column data types and spot columns that were loaded
  with the wrong type.
- Use `.describe()` on the numeric-looking columns — does anything look off?

In [3]:
df.shape
df.head()
df.tail()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Booking_ID         146 non-null    int64 
 1   Member_Name        146 non-null    object
 2   Class_Type         146 non-null    object
 3   Instructor         125 non-null    object
 4   Booking_Date       146 non-null    object
 5   Class_Time         146 non-null    object
 6   Duration_Minutes   146 non-null    object
 7   Price              135 non-null    object
 8   Payment_Method     129 non-null    object
 9   Membership_Type    124 non-null    object
 10  Attendance_Status  146 non-null    object
 11  Studio_Location    146 non-null    object
dtypes: int64(1), object(11)
memory usage: 13.8+ KB


,Booking_ID
count,146.000000
mean,1071.684932
std,40.799722
min,1001.000000
25%,1036.250000
50%,1072.500000
75%,1106.750000
max,1140.000000


### Identify Data Quality Issues

Before fixing anything, For each column,
ask yourself: does this look consistent and correctly typed?

Some things worth investigating (this list is a starting point, not a full
answer key):

- Are there columns that should be numeric but are stored as text?
- Do any categorical columns (`Class_Type`, `Payment_Method`, `Membership_Type`,
  `Attendance_Status`, `Studio_Location`) have more unique values than you'd
  expect from the real-world categories described above? (Print `.unique()` for
  each one and compare)
- Are there leading/trailing spaces or inconsistent capitalization anywhere,
  in text columns or even column values that look identical but aren't?
- Is `Booking_Date` stored in a single consistent format?
- Are there missing values, and are they always represented the same way  
- Are there any duplicate bookings?
- Do any numeric columns contain suspicious values?

In [4]:
print(df["Class_Type"].unique())
print(df["Payment_Method"].unique())
print(df["Membership_Type"].unique())
print(df["Attendance_Status"].unique())
print(df["Studio_Location"].unique())

print(df.isnull().sum())

print("Duplicates:", df.duplicated().sum())

df.info()

df.describe()

['Pilates' 'SPIN' 'Spinning' 'CROSSFIT' ' Pilates' 'zumba ' 'YOGA' 'ZUMBA'
 'HIIT' 'Zumba' 'crossfit' 'PILATES' 'Yoga ' 'yoga' 'H.I.I.T' 'Hiit'
 'Cross Fit' 'pilates' 'CrossFit' 'hiit' 'spin' 'Spin' 'Yoga']
['UNKNOWN' 'Debit Card' nan 'credit card' 'Online' 'CASH' 'Credit Card'
 'Cash ' 'online']
['VIP' ' Premium' 'Basic' 'vip' nan 'basic' 'Premium' 'unknown']
['No-show' 'ATTENDED' 'Attended' 'cancelled ' 'Cancelled' 'attended'
 'no-show']
['Uptown' 'UPTOWN' ' Westside' 'Downtown' 'downtown ' 'Westside'
 'westside']
Booking_ID            0
Member_Name           0
Class_Type            0
Instructor           21
Booking_Date          0
Class_Time            0
Duration_Minutes      0
Price                11
Payment_Method       17
Membership_Type      22
Attendance_Status     0
Studio_Location       0
dtype: int64
Duplicates: 6
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 146 entries, 0 to 145
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------

,Booking_ID
count,146.000000
mean,1071.684932
std,40.799722
min,1001.000000
25%,1036.250000
50%,1072.500000
75%,1106.750000
max,1140.000000


### Data Quality Issues Found

- Class_Type contains inconsistent capitalization, extra spaces, and different labels for the same class.
- Payment_Method contains inconsistent capitalization, extra spaces, missing values, and UNKNOWN values.
- Membership_Type contains inconsistent capitalization, leading spaces, missing values, and unknown values.
- Attendance_Status contains inconsistent capitalization and extra spaces.
- Studio_Location contains inconsistent capitalization and extra spaces.
- Missing values were found in Instructor, Price, Payment_Method, and Membership_Type.
- 6 duplicate rows were found.
- Duration_Minutes and Price are stored as object instead of numeric types.
- Booking_Date is stored as object instead of datetime.

### Handle Missing Values

- Quantify how many missing values exist in each column (remember: missing
  data isn't always `NaN` — it may be stored as an empty string or a
  placeholder like `"Unknown"` or `"N/A"`).
- For each column with missing values, think about what makes sense given
  what the column represents:
  - Does it make sense to fill it with a default value?
  - Could it be filled using information from another column or a sensible
    statistic (mean, median, or mode)?
  - Should the row be dropped instead?
- Apply your chosen strategy and briefly **justify** your decision for each
  column in a markdown note or comment.

In [6]:
df = df.replace(["UNKNOWN", "unknown"], pd.NA)

df.isnull().sum()

df["Instructor"] = df["Instructor"].fillna("Unknown")

df.isnull().sum()

,0
Booking_ID,0
Member_Name,0
Class_Type,0
Instructor,0
Booking_Date,0
Class_Time,0
Duration_Minutes,0
Price,11
Payment_Method,36
Membership_Type,39


### Missing Values Strategy

- Instructor: Missing values were replaced with "Unknown" because the instructor cannot be reliably inferred.
- Payment_Method: Missing values will be filled with the mode because it is a categorical column.
- Membership_Type: Missing values will be filled with the mode because it is a categorical column.
- Price: Missing values will be filled with the median because it is a numeric column and the median is less affected by extreme values.

### Clean Inconsistent Values

- Standardize the categorical columns (`Class_Type`, `Payment_Method`,
  `Membership_Type`, `Attendance_Status`, `Studio_Location`) so that each
  real-world category is represented by exactly one consistent label.
- Remove extra whitespace from text columns,
- Decide on a consistent capitalization style and apply it across text
  columns.
- Double-check your work by printing `.unique()` again for each cleaned
  column

In [7]:
categorical_cols = [
    "Class_Type",
    "Payment_Method",
    "Membership_Type",
    "Attendance_Status",
    "Studio_Location"
]

for col in categorical_cols:
    df[col] = df[col].str.strip()

    df["Class_Type"] = df["Class_Type"].replace({
    "PILATES": "Pilates",
    "pilates": "Pilates",
    "SPIN": "Spin",
    "spin": "Spin",
    "Spinning": "Spin",
    "CROSSFIT": "CrossFit",
    "crossfit": "CrossFit",
    "Cross Fit": "CrossFit",
    "YOGA": "Yoga",
    "yoga": "Yoga",
    "HIIT": "HIIT",
    "Hiit": "HIIT",
    "hiit": "HIIT",
    "H.I.I.T": "HIIT",
    "ZUMBA": "Zumba",
    "zumba": "Zumba"
})

df["Payment_Method"] = df["Payment_Method"].replace({
    "CASH": "Cash",
    "credit card": "Credit Card",
    "online": "Online"
})

df["Membership_Type"] = df["Membership_Type"].replace({
    "VIP": "VIP",
    "vip": "VIP",
    "basic": "Basic"
})

df["Attendance_Status"] = df["Attendance_Status"].replace({
    "ATTENDED": "Attended",
    "attended": "Attended",
    "no-show": "No-show",
    "cancelled": "Cancelled"
})

df["Studio_Location"] = df["Studio_Location"].replace({
    "UPTOWN": "Uptown",
    "westside": "Westside",
    "downtown": "Downtown"
})

for col in categorical_cols:
    print(f"{col}:")
    print(df[col].unique())
    print()

Class_Type:
['Pilates' 'Spin' 'CrossFit' 'Zumba' 'Yoga' 'HIIT']

Payment_Method:
[<NA> 'Debit Card' nan 'Credit Card' 'Online' 'Cash']

Membership_Type:
['VIP' 'Premium' 'Basic' nan <NA>]

Attendance_Status:
['No-show' 'Attended' 'Cancelled']

Studio_Location:
['Uptown' 'Westside' 'Downtown']



### Fix Data Types

- Convert `Duration_Minutes` and `Price` into proper numeric types (watch
  out for extra text or symbols)
- Convert `Booking_Date` into a proper datetime type

In [8]:
df["Duration_Minutes"] = (
    df["Duration_Minutes"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)
)

df["Duration_Minutes"] = pd.to_numeric(
    df["Duration_Minutes"],
    errors="coerce"
)

df["Price"] = (
    df["Price"]
    .astype(str)
    .str.replace(r"[^\d.]", "", regex=True)
)

df["Price"] = pd.to_numeric(
    df["Price"],
    errors="coerce"
)

df["Booking_Date"] = pd.to_datetime(
    df["Booking_Date"],
    errors="coerce"
)

df[["Duration_Minutes", "Price", "Booking_Date"]].dtypes

,0
Duration_Minutes,float64
Price,float64
Booking_Date,datetime64[ns]


### Handle Duplicates

- Check whether the dataset contains fully duplicated rows.
- Decide how to handle them
- Confirm the duplicates are gone

In [9]:
df.duplicated().sum()
df = df.drop_duplicates()
df.duplicated().sum()

np.int64(0)

### Validate the Data

After cleaning, verify that your dataset is consistent

- Re-run `.info()` and confirm every column has the expected data type.
- Confirm there are no remaining missing values
- Confirm each categorical column only contains the expected set of
  values
- Confirm there are no duplicate rows left

In [10]:
df.info()
df.isnull().sum()
for col in categorical_cols:
    print(f"{col}:")
    print(df[col].unique())
    print()

df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 144
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Booking_ID         140 non-null    int64         
 1   Member_Name        140 non-null    object        
 2   Class_Type         140 non-null    object        
 3   Instructor         140 non-null    object        
 4   Booking_Date       73 non-null     datetime64[ns]
 5   Class_Time         140 non-null    object        
 6   Duration_Minutes   140 non-null    float64       
 7   Price              129 non-null    float64       
 8   Payment_Method     105 non-null    object        
 9   Membership_Type    103 non-null    object        
 10  Attendance_Status  140 non-null    object        
 11  Studio_Location    140 non-null    object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(8)
memory usage: 14.2+ KB
Class_Type:
['Pilates' 'Spin' 'CrossFit' 'Zumba'

np.int64(0)

### Final Dataset

- Save your cleaned DataFrame to a **new** CSV file ("`cleaned_fitness_bookings.csv`")
- Do **not** overwrite or modify the original `uncleaned_fitness_bookings.csv`
  file (always keep the raw data intact for reference)


In [11]:
df.to_csv("cleaned_fitness_bookings.csv", index=False)